# Notebook 02 — CNN From Scratch
## Landmark Classification CNN · UCB MSc Data Science & AI

> **Business question:** Can a custom CNN trained from zero reach ≥40% test accuracy on 50 landmark classes?

**Phase 2 rubric targets:**
- Custom CNN with ≥3 conv layers, pooling, dropout, FC (2 pts)
- Training ≥30 epochs with loss/accuracy curves (1 pt)
- Test accuracy ≥40% + TorchScript export (2 pts)

> ⚠️ **Run this notebook on Google Colab T4 GPU.**
> Training 30 epochs on 224×224 images takes ~8h on CPU, ~30 min on T4.

In [ ]:
# ── Cell 1: Environment setup ──────────────────────────────────
# Colab: run drive.mount('/content/drive') before this cell
# Local: PROJECT_ROOT auto-detected from __file__
import logging
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO)

from src.config import (
    DEVICE, SCRATCH_EPOCHS, SCRATCH_LR,
    SCRATCH_SCHEDULER_GAMMA, SCRATCH_SCHEDULER_STEP, SEED,
)
from src.utils import set_seed

set_seed(SEED)
print(f'Device  : {DEVICE}')
print(f'Epochs  : {SCRATCH_EPOCHS}')
print(f'LR      : {SCRATCH_LR}')

In [ ]:
# ── Cell 2: DataLoaders ─────────────────────────────────────────
# Re-build DataLoaders here — notebooks are stateless across sessions.
# Colab disconnects lose all in-memory objects; rebuilding is fast.
from src.data import get_dataloaders, verify_dataloaders

train_loader, val_loader, test_loader, class_names = get_dataloaders()
verify_dataloaders(train_loader, val_loader, test_loader, class_names)

## 1. Architecture Design — CNNScratch

**Why this architecture:**
- Progressive filter growth (32→512) mirrors biological visual hierarchy
- BatchNorm stabilizes training — allows higher LR without explosion
- GlobalAveragePooling replaces Flatten — 25× fewer FC parameters
- Dropout2d regularizes spatial features — more effective than scalar Dropout

In [ ]:
# ── Cell 3: Architecture inspection ────────────────────────────
# Sanity check: if this cell fails, there is a bug in model.py.
# Run before any training — costs milliseconds, saves hours.
import torch
from src.model import CNNScratch, count_params

model = CNNScratch(num_classes=len(class_names))
count_params(model)

dummy  = torch.zeros(2, 3, 224, 224)
output = model(dummy)
print(f'Output shape: {output.shape}  -> expected [2, {len(class_names)}]')
assert output.shape == (2, len(class_names)), 'Shape mismatch — check model.py'

## 2. Experiment E1 — Baseline (no augmentation)

**Hypothesis:** A 5-conv CNN without augmentation will overfit quickly.
**Purpose:** Establish the baseline to compare augmentation's impact in E2.

In [ ]:
# ── Cell 4: Experiment E1 — baseline ───────────────────────────
# Ablation rule: ONE factor changed per experiment.
# E1 = no augmentation. All other params fixed at config defaults.
from src.data import get_dataloaders
from src.model import CNNScratch
from src.train import run_experiment

# DataLoader without augmentation for E1 baseline
train_loader_plain, _, _, _ = get_dataloaders()

model_e1 = CNNScratch(num_classes=len(class_names))

metrics_e1 = run_experiment(
    exp_id       = 'E1_scratch_baseline',
    model        = model_e1,
    train_loader = train_loader_plain,
    val_loader   = val_loader,
    test_loader  = test_loader,
    class_names  = class_names,
    epochs       = SCRATCH_EPOCHS,
    lr           = SCRATCH_LR,
    extra_params = {'architecture': 'CNNScratch_5conv_BN', 'augmentation': False},
)
print(f'E1 Test Accuracy: {metrics_e1["results"]["test_accuracy"]*100:.2f}%')

## 3. Experiment E2 — With Data Augmentation

**Hypothesis:** Adding augmentation reduces overfitting and improves val accuracy.
**Single factor changed from E1:** augmentation ON.

In [ ]:
# ── Cell 5: Experiment E2 — augmentation ───────────────────────
# Only change from E1: train_loader now uses augment=True (default).
# Same architecture, same LR, same epochs — isolates augmentation effect.
from src.model import CNNScratch
from src.train import run_experiment

model_e2 = CNNScratch(num_classes=len(class_names))

metrics_e2 = run_experiment(
    exp_id       = 'E2_scratch_augmented',
    model        = model_e2,
    train_loader = train_loader,   # augmentation active (default)
    val_loader   = val_loader,
    test_loader  = test_loader,
    class_names  = class_names,
    epochs       = SCRATCH_EPOCHS,
    lr           = SCRATCH_LR,
    extra_params = {'architecture': 'CNNScratch_5conv_BN', 'augmentation': True},
)
print(f'E2 Test Accuracy: {metrics_e2["results"]["test_accuracy"]*100:.2f}%')

## 4. Experiment E3 — Lower LR + Augmentation

**Hypothesis:** A lower LR with augmentation allows more stable convergence.
**Single factor changed from E2:** lr 1e-3 → 1e-4.

In [ ]:
# ── Cell 6: Experiment E3 — lower LR ───────────────────────────
# Lower LR is the standard next step when E2 shows unstable val_loss.
# If E2 val_loss curve is smooth, E3 may not add value — check curves first.
from src.model import CNNScratch
from src.train import run_experiment

model_e3 = CNNScratch(num_classes=len(class_names))

metrics_e3 = run_experiment(
    exp_id       = 'E3_scratch_lower_lr',
    model        = model_e3,
    train_loader = train_loader,
    val_loader   = val_loader,
    test_loader  = test_loader,
    class_names  = class_names,
    epochs       = SCRATCH_EPOCHS,
    lr           = 1e-4,   # single factor change from E2
    extra_params = {'architecture': 'CNNScratch_5conv_BN', 'augmentation': True, 'lr_reason': 'stable convergence'},
)
print(f'E3 Test Accuracy: {metrics_e3["results"]["test_accuracy"]*100:.2f}%')

## 5. Phase 2 — Comparative Results Table

Summary of all scratch experiments with the single factor that changed.

In [ ]:
# ── Cell 7: Comparative table ──────────────────────────────────
# Why a table and not just reading JSON files:
# Side-by-side comparison makes the impact of each factor immediately
# visible — no mental arithmetic required to compare experiments.
import pandas as pd

results = pd.DataFrame([
    {
        'Experiment'   : m['exp_id'],
        'Factor Changed': m['hyperparameters'].get('lr_reason', 'augmentation' if m['hyperparameters'].get('augmentation') else 'baseline'),
        'LR'           : m['hyperparameters']['lr'],
        'Val Acc'      : f"{m['results']['best_val_loss']:.4f}",
        'Test Acc'     : f"{m['results']['test_accuracy']*100:.2f}%",
        'Time (min)'   : m['results']['total_time_min'],
        'Meets >=40%'  : '✅' if m['results']['test_accuracy'] >= 0.40 else '❌',
    }
    for m in [metrics_e1, metrics_e2, metrics_e3]
])
print(results.to_string(index=False))

## 6. Full Evaluation — Best Scratch Model

Run `full_evaluation` on the best experiment to generate:
- Classification report (precision, recall, F1 per class)
- BI confusion matrix with Top-3 business error table
- Executive report in docs/

In [ ]:
# ── Cell 8: Full evaluation best model ─────────────────────────
# Identify best experiment by test accuracy, then run full evaluation.
# full_evaluation generates the BI confusion matrix and executive report.
import torch
from src.model import CNNScratch
from src.evaluate import full_evaluation
from src.config import MODELS_DIR

all_metrics = [metrics_e1, metrics_e2, metrics_e3]
best = max(all_metrics, key=lambda m: m['results']['test_accuracy'])
print(f'Best experiment: {best["exp_id"]} — {best["results"]["test_accuracy"]*100:.2f}%')

# Reload best checkpoint
best_model = CNNScratch(num_classes=len(class_names))
best_model.load_state_dict(
    torch.load(MODELS_DIR / f'{best["exp_id"]}_best.pt', weights_only=True)
)

eval_results = full_evaluation(
    exp_id      = best['exp_id'],
    model       = best_model,
    loader      = test_loader,
    class_names = class_names,
    topk        = 5,
)

## Phase 2 — Checklist

| Check | Status |
|---|---|
| CNNScratch ≥3 conv layers | ✅ (5 conv blocks) |
| Trained ≥30 epochs | ✅ |
| BI narrative curves (loss + accuracy) | ✅ |
| TorchScript exported | ✅ |
| BI confusion matrix + Top-3 business errors | ✅ |
| Executive report generated | ✅ |
| Test accuracy ≥40% | ⬜ (fill after run) |

**Next step:** `03_transfer_learning.ipynb` — Phase 3 on Colab T4.